# 🚗 Insurance Pricing Engine — visão geral

Motor de pricing de seguros pelo método atuarial clássico (**frequência × severidade → prêmio puro**), com uma camada de **explicabilidade (SHAP)** comparando GLM (transparente por construção) e GBM (mais preciso). Projeto flagship de portfólio em **pricing / risco de seguros**.

Este notebook é o **índice navegável** + um **glossário** para quem é de fora do mundo de seguros. A documentação completa está no [README](../README.md) e no Word versionado (`docs/`).

## Como navegar

| Notebook | O que você vê |
|---|---|
| [01_eda](01_eda.ipynb) | De onde vêm os dados, distribuições, drivers de risco |
| [02_modeling](02_modeling.ipynb) | GLM freq/sev → prêmio puro → GLM vs GBM + SHAP |
| [03_diagnostics](03_diagnostics.ipynb) | Relatividades, observado×previsto, lift chart, Gini/Lorenz |

O código reutilizável (o **motor**) vive em `src/pricing/`; os **runners** são os `run_*.py`. Os notebooks contam a história importando o motor, sem duplicar lógica. A demo interativa é o `streamlit_app.py` (`python -m streamlit run streamlit_app.py`).

## Resultados em destaque (v1)

| Métrica | Resultado |
|---|---|
| Frequência (GLM Poisson) | **−2,97%** de deviance vs média ingênua (678k apólices) |
| Prêmio puro | **Tweedie direto calibra a carteira (1,04)**; freq×sev erra o total em **+33%** |
| GLM vs GBM | GBM **+5,15%** mais preciso, mas o **SHAP mostra os mesmos drivers** → ganho confiável |

![Importância SHAP do GBM](../reports/figures/shap_gbm_frequency.png)

## Roadmap (flagship modular)

- **v1** — Núcleo GLM + XAI *(✅ concluída)*
- **v2** — Elasticidade & otimização de prêmio (base ES, com validação temporal)
- **v3** — MLOps (MLflow, Docker, drift PSI, CI)
- **v4** — Score territorial Brasil (IBGE/CNEFE/CEP + SUSEP)

## Como rodar

Detalhes no [README](../README.md). Em resumo:

```bash
pip install -r requirements.txt
python download_datasets.py && python run_etl.py        # bases -> data/raw -> data/processed
python run_frequency_baseline.py                        # baseline de frequência
python run_pure_premium.py                              # prêmio puro
python run_gbm_vs_glm.py                                # GLM vs GBM + SHAP
python -m streamlit run streamlit_app.py                # demo interativa
```

Os notebooks consomem `data/processed/` (gerado por `run_etl.py`).

---
# 📖 Glossário & contexto
*Para quem é de fora do mundo de seguros — o macro do universo, os tratamentos, os modelos e as métricas.*

## 1. O universo de seguros (o macro)

- Uma **seguradora** vende proteção: o cliente paga um **prêmio** e, se sofrer um evento coberto (**sinistro** — batida, roubo, etc.), recebe a indenização.
- O negócio funciona por **mutualização**: muitos pagam, poucos sinistram; o prêmio de todos cobre os sinistros de alguns + despesas + margem.
- O risco **varia por perfil**. Cobrar o mesmo de todos é injusto e perigoso (**seleção adversa**: os bons riscos vão para a concorrência mais barata e sobram os ruins). Por isso existe **pricing**: estimar o risco de cada perfil e precificar de acordo.
- **Onde o Data Science entra:** estimar o custo esperado de sinistros por perfil (o *prêmio puro*) com modelos estatísticos — e de forma **regulada**, porque o regulador (no Brasil, a **SUSEP**) exige que o preço seja justificável e não-discriminatório. Daí a importância de modelos explicáveis.

## 2. Conceitos-chave de pricing

| Termo | O que é |
|---|---|
| **Sinistro** (claim) | o evento coberto que gera pagamento |
| **Apólice** (policy) | o contrato de seguro de um cliente |
| **Exposição** (exposure) | fração do ano em que a apólice esteve coberta (0,5 = meio ano). Serve de base justa de comparação |
| **Frequência** | nº de sinistros por ano de exposição (ex.: 0,10 = ~1 sinistro a cada 10 anos) |
| **Severidade** | custo médio de um sinistro quando ele ocorre |
| **Prêmio puro** | custo esperado de sinistros = **frequência × severidade** |
| **Prêmio comercial** | prêmio puro **+ despesas + margem + impostos** — o que o cliente realmente paga |
| **Bônus-malus** | ajuste pelo histórico: bom motorista ganha desconto (bônus); quem sinistra paga mais (malus). Valor **>100 = penalizado** |
| **Loss ratio** | sinistros ÷ prêmios — termômetro da saúde da carteira |

## 3. Tratamento dos dados (ETL) — o que foi feito

- **Fonte:** freMTPL2 (apólices de auto de responsabilidade civil francesas), em 2 tabelas — *frequência* (1 linha por apólice) e *severidade* (1 linha por sinistro) — unidas por `IDpol`.
- **Limpeza:** `ClaimNb` limitado a 4 e `Exposure` a 1 ano (apara outliers); apólices com exposição zero são removidas (não informam taxa).
- **Junção e derivação:** somamos o valor dos sinistros por apólice; criamos `Frequency = ClaimNb/Exposure` e `PurePremium = ClaimAmount/Exposure`.
- **Materialização:** snapshot estável em `data/processed/` (parquet) via `run_etl.py` — a *camada gold* que os notebooks e o app consomem (em vez de recalcular do zero).

## 4. Os modelos

| Modelo | Para quê | Distribuição | Por quê |
|---|---|---|---|
| **GLM Poisson** | frequência | Poisson | conta eventos raros; a exposição entra como *offset* |
| **GLM Gamma** | severidade | Gamma | custos positivos e assimétricos (cauda à direita) |
| **Tweedie** | prêmio puro direto | Tweedie (1<p<2) | mistura Poisson+Gamma: trata os muitos zeros **e** a cauda num modelo só |
| **GBM** | desafiante (frequência) | Poisson | *machine learning* (árvores): capta não-linearidades/interações; mais preciso, menos transparente |
| **SHAP** | explicar o GBM | — | decompõe cada previsão nos fatores que a empurraram → *abre a caixa-preta* |

**A grande ideia:** o **GLM é transparente por construção** (cada fator é um multiplicador auditável); o **GBM é mais preciso** mas opaco, então o **SHAP** o audita. O tema central do projeto é o trade-off **precisão × interpretabilidade** num contexto regulado.

## 5. As métricas

| Métrica | O que mede | Como ler |
|---|---|---|
| **Deviance** (Poisson/Gamma/Tweedie) | erro do modelo na distribuição certa | **menor = melhor**; comparado sempre vs uma *média ingênua* |
| **Calibração de carteira** | total previsto ÷ total real de sinistros | ideal **≈ 1,0** (não sub/superestima o caixa) |
| **Lift chart** | poder de separar bom de mau risco | curva **ascendente** por decil de risco previsto |
| **Gini / Lorenz** | ranqueamento de risco num número só | **maior = melhor** ordenação |
| **Relatividades** (exp do coef.) | efeito de cada fator no GLM | **>1 encarece, <1 barateia** |

*Por que não usar só "acurácia"?* Porque o alvo é contagem/custo com muitos zeros e cauda pesada — a *deviance* na distribuição correta e a *calibração* dizem muito mais sobre um modelo de pricing do que um R² ou acurácia ingênuos.